# Figure 5

Paper panels with figure-specific PDF and CSV exports. Run from top to bottom. Original analysis notebooks are preserved.

## Setup

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import contextmanager
from IPython.display import IFrame, display
from matplotlib.transforms import Bbox
import itertools
import os
import shutil
import pandas as pd
import KinematicPlot as kp
from group_config_new import build_groups
from survival_stats_runner import SurvivalStatsRunner

# All persistent exports belong to this paper figure, with one folder per panel.
ROOT = Path.cwd()
if not (ROOT / "KinematicPlot.py").is_file():
    raise RuntimeError("Run this notebook from the repository root.")
FIGURE_NUMBER = 5
NOTEBOOK_OUTPUT_DIR = ROOT / "Figures" / f"Figure{FIGURE_NUMBER}"
SC_DATA_DIR = ROOT / "SC data"
N_PERM = 20000
plotter = kp.PlotCreator()
stats_runner = SurvivalStatsRunner(tau=0.71, random_state=0, platform_offset=0.03, radius=0.07, fps=250)

def output_folder(*parts):
    # Create output folders only when the notebook is executed.
    folder = NOTEBOOK_OUTPUT_DIR.joinpath(*parts)
    folder.mkdir(parents=True, exist_ok=True)
    return folder

def panel_prefix(panel, name):
    return output_folder(f"Figure{panel}") / f"Figure{panel}_{name}"

def show_pdf(path, width=950, height=700):
    display(IFrame(src=Path(path).relative_to(ROOT).as_posix(), width=width, height=height))

@contextmanager
def working_directory(folder):
    # Preserve the working directory for legacy notebook loader calls.
    previous = Path.cwd()
    os.chdir(folder)
    try:
        yield
    finally:
        os.chdir(previous)

In [4]:
# Preserve the current optogenetic settings: 100 ms gap and 50% invalid tolerance.
OPTO_QC_ERROR_MAX = 30
OPTO_QC_SCORE_MIN = 0.8
OPTO_QC_MIN_CAMERAS = 2
OPTO_QC_MAX_INTERP_GAP_S = 0.1
OPTO_QC_MAX_INVALID_FRACTION = 0.5
OPTO_QC_MIN_VALID_FRACTION = 1.0 - OPTO_QC_MAX_INVALID_FRACTION

# Smooth after QC/interpolation with the utilities exponential moving average.
OPTO_ANGLE_SMOOTH = True
OPTO_ANGLE_SMOOTH_ALPHA = 0.4

OPTO_ANGLE_QC_KWARGS = dict(
    apply_tracking_qc=True,
    min_cameras=OPTO_QC_MIN_CAMERAS,
    max_interp_gap_s=OPTO_QC_MAX_INTERP_GAP_S,
    min_valid_fraction=OPTO_QC_MIN_VALID_FRACTION,
    error_max=OPTO_QC_ERROR_MAX,
    score_min=OPTO_QC_SCORE_MIN,
    smooth_angle=OPTO_ANGLE_SMOOTH,
    smooth_alpha=OPTO_ANGLE_SMOOTH_ALPHA,
)

## Optogenetic groups, colors, and selected comparisons

In [6]:
chr_low_keys = [
    "ADxChr-400uW",
    "IavxChr-400uW",
    "HP2xChr-400uW",
    "TaCSxCHR-400uW",
    "AllCSxChr-400uW",
    "ANxCHR-400uW",
]
chr_med_keys = [
    "IAVxCHR-4mW",
    "HP2xCHR-4mW",
    "TaBriLexAR-4mW",
    "TaCSxCHR-4mW",
    "CSS0048xCHR-4mW",
    "BiCSxCHR-4mW",
    "BiCS-HaltxCHR-4mW",
    "ANxChr-4mW",
]
chr_high_keys = [
    "IAVxCHR-12mW",
    "HP2xChr-12mW",
    "TaBriLexAR-12mW",
    "TaCSxCHR-12mW",
    "CSS0048xCHR-12mW",
    "BiCS-HaltWgxCHR-12mW",
    "BICSxCHR-12mW",
    "BICSHALTxCHR-12mW",
    "CSS0021xCHR-12mW",
    "ANxCHR-12mW",
]
gtacr_keys = [
    "WT_Green",
    "LexA_Br",
    "MTGal4",
    "IavxGTACR",
    "CSS0048xGTACR",
    "CSS0021xGTACR",
    "ANxGTACR"
]

chr_intensity_blocks = {
    "low": {"lp_label": "4A", "ll_label": "4D", "keys": chr_low_keys},
    "medium": {"lp_label": "4B", "ll_label": "4E", "keys": chr_med_keys},
    "high": {"lp_label": "4C", "ll_label": "4F", "keys": chr_high_keys},
}

chr_lp_change_keys = [
    "ADxChr-400uW",
    "IavxChr-400uW", "IAVxCHR-4mW", "IAVxCHR-12mW",
    "HP2xChr-400uW", "HP2xCHR-4mW", "HP2xChr-12mW",
    "AllCSxChr-400uW",
    "BiCSxCHR-4mW", "BICSxCHR-12mW",
    "BiCS-HaltxCHR-4mW", "BICSHALTxCHR-12mW",
    "BiCS-HaltWgxCHR-12mW",
    "CSS0048xCHR-4mW", "CSS0048xCHR-12mW",
    "CSS0021xCHR-12mW",
    "TaCSxCHR-400uW", "TaCSxCHR-4mW", "TaCSxCHR-12mW",
    "TaBriLexAR-4mW", "TaBriLexAR-12mW",
]

chrimson_selected_keys = [
    "ADxChr-400uW",
    "AllCSxChr-400uW",
    "CSS0048xCHR-12mW",
    "CSS0021xCHR-12mW",
    "TaBriLexAR-4mW",
]
chrimson_selected_colors = [
    "black",
    "green",
    "blue",
    "brown",
    "red",
]

gtacr_colors = {
    "WT_Green": "black",
    "LexA_Br": "green",
    "MTGal4": "blue",
    "IavxGTACR": "brown",
    "CSS0048xGTACR": "red",
    "CSS0021xGTACR": "orange",
    "ANxGTACR": "yellow",
}

chr_lp_intensity_colors = {"low": "#F4A3A3", "medium": "#D73027", "high": "#7F0000"}

# Figure 5 excludes every AN group while retaining the existing controls.
chr_intensity_blocks = {level: {"keys": [key for key in block["keys"] if not key.startswith("AN")]} for level, block in chr_intensity_blocks.items()}
gtacr_keys = [key for key in gtacr_keys if key != "ANxGTACR"]
required_keys = list(dict.fromkeys([key for block in chr_intensity_blocks.values() for key in block["keys"]] + gtacr_keys))
groups = build_groups(group_keys=required_keys, skip_missing=False, require_kinematics=False)
for group in groups.values():
    group.initialize_manual_data()
    group.filter_opto_data(min_trial_num=8)

## Figure 5A - CsChrimson OFF/ON landing probability

In [8]:
# The generic paired LP function saves the sign-flip test and significance bracket.
for intensity, block in chr_intensity_blocks.items():
    for key in block["keys"]:
        prefix = panel_prefix("5A", key + "_LP_ON_OFF")
        plotter.plot_LP_summary_light_from_group(groups[key], str(prefix), color=chr_lp_intensity_colors[intensity], min_trial_num=8, n_perm=N_PERM)
        show_pdf(str(prefix) + "-LP.pdf")

## Figure 5B - Selected CsChrimson ON latency

In [10]:
selected_on_ll_data = []
for group_key in chrimson_selected_keys:
    # Extract ON-trial latency rows from normally initialized optogenetic metadata.
    selected_on_ll_data.append(
        plotter.get_opto_on_ll_data(
            group_info=groups[group_key],
            tau=0.71,
            min_trial_num=8,
        )
    )

km_out = output_folder("Figure5B")
km_prefix = km_out / "Figure5B_selected_CsChrimson_ON_landing_latency_KM"
selected_on_km_stats_df, selected_on_fly_rmst_df = plotter.plot_kmc_and_unpaired_rmst_perm(
    data_list=selected_on_ll_data,
    file_name=str(km_prefix),
    tau=0.71,
    n_perm=N_PERM,
    random_state=0,
    colors=chrimson_selected_colors,
    invert_curve=True,
    control_group="ADxCHR-400uW",
)
display(selected_on_km_stats_df)
display(selected_on_fly_rmst_df.head())
show_pdf(f"{km_prefix}-KMC.pdf")

,comparison,test,metric,group_a,group_b,N_a,mean_a,std_a,N_b,mean_b,std_b,n_a,n_b,mean_diff_b_minus_a,p_value,n_perm,n_pairwise_comparison,tau
0,ADxCHR-400uW vs ALLCSxCHR-400uW,pairwise_flywise_rmst_unpaired_permutation,landing_latency_rmst,ADxCHR-400uW,ALLCSxCHR-400uW,15,0.71,2.298380e-16,14,0.580900,0.202891,207,193,-0.129100,0.000900,20000,4,0.71
1,ADxCHR-400uW vs CSS0048xCHR-12mW,pairwise_flywise_rmst_unpaired_permutation,landing_latency_rmst,ADxCHR-400uW,CSS0048xCHR-12mW,15,0.71,2.298380e-16,18,0.664681,0.086356,207,257,-0.045319,0.048198,20000,4,0.71
2,ADxCHR-400uW vs CSS0021xCHR-12mW,pairwise_flywise_rmst_unpaired_permutation,landing_latency_rmst,ADxCHR-400uW,CSS0021xCHR-12mW,15,0.71,2.298380e-16,10,0.699827,0.021729,207,149,-0.010173,0.117644,20000,4,0.71
3,ADxCHR-400uW vs TaBriLexAR-4mW,pairwise_flywise_rmst_unpaired_permutation,landing_latency_rmst,ADxCHR-400uW,TaBriLexAR-4mW,15,0.71,2.298380e-16,7,0.093686,0.054803,207,101,-0.616314,0.000050,20000,4,0.71


,Group,Fly#,RMST,n_trials,n_events,event_rate
0,ADxCHR-400uW,1,0.71,15,0,0.0
1,ADxCHR-400uW,2,0.71,15,0,0.0
2,ADxCHR-400uW,3,0.71,15,0,0.0
3,ADxCHR-400uW,4,0.71,15,0,0.0
4,ADxCHR-400uW,5,0.71,10,0,0.0


## Figure 5C - Selected CsChrimson R-mFT angle traces

In [12]:
out = output_folder("Figure5C")
# Save this figure as an R-mFT-only trace because the angle list below contains one joint definition.
pdf = out / "Figure5C_selected_CsChrimson_ON_RmFT_angle_traces.pdf"
selected_angle_result = plotter.plot_selected_chrimson_angle_traces(
    groups={key: groups[key] for key in chrimson_selected_keys},
    angles=[
        ["R-mCT", "R-mFT", "R-mTT"],
    ],
    file_name=str(pdf.with_suffix("")),
    start=-0.5,
    end=3,
    condition="ON",
    colors=chrimson_selected_colors,
    show_sem=True,
    qc_start=0,
    qc_end=2.0,
    **OPTO_ANGLE_QC_KWARGS,
)
if len(selected_angle_result) == 5:
    selected_angle_fig, selected_angle_axes, selected_angle_summary_df, selected_angle_qc_df, selected_angle_skipped_df = selected_angle_result
    display(selected_angle_qc_df.head())
    display(selected_angle_skipped_df.head())
    if not selected_angle_skipped_df.empty:
        display(
            selected_angle_skipped_df
            .groupby(["Plot_Label", "Joint", "Reason"], as_index=False)
            .size()
            .sort_values(["Plot_Label", "Joint", "Reason"])
        )
elif len(selected_angle_result) == 4:
    selected_angle_fig, selected_angle_axes, selected_angle_summary_df, selected_angle_qc_df = selected_angle_result
    selected_angle_skipped_df = pd.DataFrame()
    display(selected_angle_qc_df.head())
else:
    selected_angle_fig, selected_angle_axes, selected_angle_summary_df = selected_angle_result
    selected_angle_qc_df = pd.DataFrame()
    selected_angle_skipped_df = pd.DataFrame()
display(selected_angle_summary_df)
show_pdf(pdf, height=850)

,Joint,Angle_Definition,QC_Passed,QC_Exclusion_Reason,Valid_Frame_Fraction,Invalid_Frame_Fraction,Max_Invalid_Gap_Frames,Interpolated_Frame_Count,Max_Interp_Gap_Frames,Index,Fly#,Trial#,Group_Name,Plot_Label,Condition
0,R-mFT,R-mCT|R-mFT|R-mTT,True,,1.0,0.0,0,0,25,"(1, 3)",1,3,ADxCHR-400uW,ADxChr-400uW,ON
1,R-mFT,R-mCT|R-mFT|R-mTT,True,,1.0,0.0,0,0,25,"(1, 4)",1,4,ADxCHR-400uW,ADxChr-400uW,ON
2,R-mFT,R-mCT|R-mFT|R-mTT,True,,1.0,0.0,0,0,25,"(1, 5)",1,5,ADxCHR-400uW,ADxChr-400uW,ON
3,R-mFT,R-mCT|R-mFT|R-mTT,True,,1.0,0.0,0,0,25,"(1, 8)",1,8,ADxCHR-400uW,ADxChr-400uW,ON
4,R-mFT,R-mCT|R-mFT|R-mTT,True,,1.0,0.0,0,0,25,"(1, 10)",1,10,ADxCHR-400uW,ADxChr-400uW,ON


,Group_Name,Index,Fly#,Trial#,Joint,Angle_Definition,Reason,Alignment_Frame,Trace_Start_Frame,Trace_End_Frame,Requested_Start_s,Requested_End_s,Finite_Frame_Count,Plot_Label,Condition
0,ADxCHR-400uW,"(13, 20)",13,20,R-mFT,R-mCT|R-mFT|R-mTT,fewer_than_two_finite_frames,750,625,1499,-0.5,3,0,ADxChr-400uW,ON
1,ADxCHR-400uW,"(13, 21)",13,21,R-mFT,R-mCT|R-mFT|R-mTT,fewer_than_two_finite_frames,750,625,1499,-0.5,3,0,ADxChr-400uW,ON
2,ALLCSxCHR-400uW,"(1, 3)",1,3,R-mFT,R-mCT|R-mFT|R-mTT,fewer_than_two_finite_frames,750,625,1499,-0.5,3,0,AllCSxChr-400uW,ON
3,ALLCSxCHR-400uW,"(1, 4)",1,4,R-mFT,R-mCT|R-mFT|R-mTT,fewer_than_two_finite_frames,750,625,1499,-0.5,3,0,AllCSxChr-400uW,ON
4,ALLCSxCHR-400uW,"(1, 5)",1,5,R-mFT,R-mCT|R-mFT|R-mTT,fewer_than_two_finite_frames,750,625,1499,-0.5,3,0,AllCSxChr-400uW,ON


,Plot_Label,Joint,Reason,size
0,ADxChr-400uW,R-mFT,fewer_than_two_finite_frames,2
1,AllCSxChr-400uW,R-mFT,fewer_than_two_finite_frames,26
2,CSS0021xCHR-12mW,R-mFT,fewer_than_two_finite_frames,4
3,CSS0048xCHR-12mW,R-mFT,fewer_than_two_finite_frames,48
4,TaBriLexAR-4mW,R-mFT,fewer_than_two_finite_frames,41


,Group,Plot_Label,Condition,Joint,Trace_Type,n_trials,total_trials,valid_total_label,start,end,qc_start,qc_end,Apply_Tracking_QC
0,ADxCHR-400uW,ADxChr-400uW,ON,R-mFT,R-mFT,205,207,205/207,-0.5,3,0,2.0,True
1,ALLCSxCHR-400uW,AllCSxChr-400uW,ON,R-mFT,R-mFT,167,193,167/193,-0.5,3,0,2.0,True
2,CSS0048xCHR-12mW,CSS0048xCHR-12mW,ON,R-mFT,R-mFT,209,257,209/257,-0.5,3,0,2.0,True
3,CSS0021xCHR-12mW,CSS0021xCHR-12mW,ON,R-mFT,R-mFT,145,149,145/149,-0.5,3,0,2.0,True
4,TaBriLexAR-4mW,TaBriLexAR-4mW,ON,R-mFT,R-mFT,60,101,60/101,-0.5,3,0,2.0,True


## Figure 5D - GtACR OFF/ON landing probability

In [14]:
for key in gtacr_keys:
    prefix = panel_prefix("5D", key + "_LP_ON_OFF")
    # The plotting call records the paired sign-flip statistics with the figure.
    plotter.plot_LP_summary_light_from_group(groups[key], str(prefix), color="green", n_perm=N_PERM)
    show_pdf(str(prefix) + "-LP.pdf")

## Figure 5E - GtACR OFF/ON landing latency

In [16]:
ll_rows = []
for key in gtacr_keys:
    prefix = panel_prefix("5E", key + "_LL_ON_OFF")
    plotter.plot_KM_curve_from_groups(groups=[groups[key]], file_name=str(prefix), colors=["black", "green"], opto=True)
    # Preserve the old notebook's paired fly-RMST analysis alongside each KM plot.
    result, _ = stats_runner.analyze_landing_opto(groups[key], out_prefix=str(prefix), n_perm=N_PERM)
    ll_rows.append(result)
    show_pdf(str(prefix) + "-LL-KMC-flipped.pdf")
pd.concat(ll_rows, ignore_index=True).to_csv(str(panel_prefix("5E", "GtACR")) + "_paired_fly_RMST_stats.csv", index=False)